In [1]:
import pandas as pd
import requests
import re
import os
from tqdm.auto import tqdm
from sklearn.metrics import cohen_kappa_score

# ==========================================
# 0. PENGECEKAN & PENYESUAIAN PATH LOKAL
# ==========================================
# Menggunakan '../' karena posisi notebook berada di dalam folder 'notebook'
train_path = '../dataset/train.csv'
test_path = '../dataset/test.csv'
output_dir = '../outputs'
submission_path = os.path.join(output_dir, 'submission.csv')

if not os.path.exists(train_path) or not os.path.exists(test_path):
    raise FileNotFoundError(
        "File dataset tidak ditemukan! Pastikan folder 'dataset' yang berisi "
        "train.csv dan test.csv berada sejajar di luar folder 'notebook' Anda saat ini."
    )

# Membuat folder 'outputs' secara otomatis jika belum ada
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# ==========================================
# 1. PERSIAPAN DATASET
# ==========================================
print("Loading dataset...")
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Mengambil 20 sampel acak
experiment_df = train_df.sample(20, random_state=42).copy()

# ==========================================
# 2. FUNGSI PANGGILAN LLM (OLLAMA LOKAL)
# ==========================================
def request_ollama_score(prompt_text, model_name="llama3"):
    api_url = "http://localhost:11434/api/generate"
    payload = {
        "model": model_name,
        "prompt": prompt_text,
        "stream": False,
        "temperature": 0.0 # Strict mode
    }
    
    try:
        response = requests.post(api_url, json=payload, timeout=30)
        response.raise_for_status()
        reply_text = response.json().get('response', '')
        
        # Ekstraksi angka skor (1 sampai 6)
        extracted_scores = re.findall(r'[1-6]', reply_text)
        if extracted_scores:
            return int(extracted_scores[0])
        return 3 # Default jika tidak menemukan angka
    except requests.exceptions.ConnectionError:
        print("\nError: Tidak dapat terhubung ke Ollama. Pastikan aplikasi Ollama sudah berjalan!")
        return 3
    except Exception as e:
        return 3

# ==========================================
# 3. ZERO-SHOT LEARNING
# ==========================================
print("\n--- Running Zero-Shot Evaluation ---")
zero_shot_results = []

zero_shot_template = """As an expert human evaluator, grade the following student essay on a scale of 1 to 6.
Rule: Output strictly a single integer between 1 and 6. No explanations.
Essay content:
"{essay_text}"

Score:"""

for _, row in tqdm(experiment_df.iterrows(), total=len(experiment_df)):
    formatted_prompt = zero_shot_template.format(essay_text=row['full_text'])
    score = request_ollama_score(formatted_prompt)
    zero_shot_results.append(score)

experiment_df['pred_zero_shot'] = zero_shot_results

# ==========================================
# 4. FEW-SHOT LEARNING
# ==========================================
print("\n--- Running Few-Shot Evaluation ---")
bad_essay = train_df[train_df['score'] == 1].iloc[0]['full_text'][:300] + "..."
good_essay = train_df[train_df['score'] == 6].iloc[0]['full_text'][:300] + "..."

few_shot_template = f"""As an expert human evaluator, grade the following student essay on a scale of 1 to 6.
Rule: Output strictly a single integer between 1 and 6. No explanations.

Here are reference examples:
[Example Score 1]
Text: "{bad_essay}"
Score: 1

[Example Score 6]
Text: "{good_essay}"
Score: 6

Now evaluate this target essay:
Text: "{{essay_text}}"
Score:"""

few_shot_results = []
for _, row in tqdm(experiment_df.iterrows(), total=len(experiment_df)):
    formatted_prompt = few_shot_template.format(essay_text=row['full_text'])
    score = request_ollama_score(formatted_prompt)
    few_shot_results.append(score)

experiment_df['pred_few_shot'] = few_shot_results

# ==========================================
# 5. EVALUASI METRIK (QWK)
# ==========================================
qwk_zero = cohen_kappa_score(experiment_df['score'], experiment_df['pred_zero_shot'], weights='quadratic')
qwk_few = cohen_kappa_score(experiment_df['score'], experiment_df['pred_few_shot'], weights='quadratic')

print(f"\n[Evaluation Results]")
print(f"Zero-Shot QWK : {qwk_zero:.4f}")
print(f"Few-Shot QWK  : {qwk_few:.4f}")

# ==========================================
# 6. KAGGLE SUBMISSION (MENGGUNAKAN FEW-SHOT)
# ==========================================
print("\n--- Generating Kaggle Submission ---")
final_predictions = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    prompt = few_shot_template.format(essay_text=row['full_text'])
    final_predictions.append(request_ollama_score(prompt))

submission = pd.DataFrame({
    'essay_id': test_df['essay_id'],
    'score': final_predictions
})

# Menyimpan ke folder outputs yang berada di luar folder notebook
submission.to_csv(submission_path, index=False)
print(f"Saved to '{submission_path}'!")

Loading dataset...

--- Running Zero-Shot Evaluation ---


  0%|          | 0/20 [00:00<?, ?it/s]


--- Running Few-Shot Evaluation ---


  0%|          | 0/20 [00:00<?, ?it/s]


[Evaluation Results]
Zero-Shot QWK : 0.1667
Few-Shot QWK  : -0.0455

--- Generating Kaggle Submission ---


  0%|          | 0/3 [00:00<?, ?it/s]

Saved to '../outputs\submission.csv'!
